<img src=../images/gdd-logo.png width=300px align=right>

# Combining Datasets

In this notebook we'll ways to combine datasets: concatenating, merging and joining.

- [Concatenating datasets](#c)
    - <mark>[Exercise: Concatenating](#e0) </mark>
- [Merging datasets](#mj)
    - <mark>[Assignment](#e1) </mark>

<a id='c'></a>
## Concatenating datasets

This time, let's imagine you didn't recieve the `chickweight` dataset in its entirety. 

Instead you recieve **four separate datasets**, one for each diet.

In [1]:
import pandas as pd

In [2]:
diet_1 = pd.read_csv('../data/diet_1.csv')
diet_2 = pd.read_csv('../data/diet_2.csv')
diet_3 = pd.read_csv('../data/diet_3.csv')
diet_4 = pd.read_csv('../data/diet_4.csv')

You can see from the below that the dataframe `diet_1` has in fact only got information where `diet` is equal to 1.

In [3]:
diet_1['diet'].unique()

array([1])

To recreate the `chickweight` dataset, you would need to **vertically stack** these datasets on top of one another to do so. 

The `pd.concat()` method can be used to do this.

In [4]:
chickweight_concat = pd.concat([diet_1, diet_2, diet_3, diet_4])
chickweight_concat['diet'].unique()

### acts liek a union

array([1, 2, 3, 4])

<a id='e0'></a>
## <mark> Exercise: Concatenating </mark>

1. The above results in the our original dataset, but the indexes are not the same if you read the full data in. Alter the code so that the resulting index is the default value `0 - 578`

<details>
<summary><font style="color:blue;font-weight:bold">SHOW HINT</font></summary>
  
There are two ways to do this:
    
1. Use the method `.reset_index()` on the resulting DataFrame.
2. Use a parameter in the [pd.concat](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html)
    
</details>

In [13]:
chickweight_concat = pd.concat([diet_1, diet_2, diet_3, diet_4])
chickweight_concat['diet'].unique()
chickweight_concat.reset_index(drop=True)

,rownum,weight,time,chick,diet
0,1,42,0,1,1
1,2,51,2,1,1
2,3,59,4,1,1
3,4,64,6,1,1
4,5,76,8,1,1
...,...,...,...,...,...
573,574,175,14,50,4
574,575,205,16,50,4
575,576,234,18,50,4
576,577,264,20,50,4


2. Concatenate `diet_1` and `diet_2` together horizontally. Why do we get the missing `NaN` values?

<details>
<summary><font style="color:blue;font-weight:bold">SHOW HINT</font></summary>
  
You will need to use the `axis=` parameter to specify you want to concatenate across the columns (horizontally).
    
</details>

In [10]:
pd.concat([diet_1, diet_2], axis=1)

,rownum,weight,time,chick,diet,rownum,weight,time,chick,diet
0,1,42,0,1,1,221.0,40.0,0.0,21.0,2.0
1,2,51,2,1,1,222.0,50.0,2.0,21.0,2.0
2,3,59,4,1,1,223.0,62.0,4.0,21.0,2.0
3,4,64,6,1,1,224.0,86.0,6.0,21.0,2.0
4,5,76,8,1,1,225.0,125.0,8.0,21.0,2.0
...,...,...,...,...,...,...,...,...,...,...
215,216,89,14,20,1,NaN,NaN,NaN,NaN,NaN
216,217,98,16,20,1,NaN,NaN,NaN,NaN,NaN
217,218,107,18,20,1,NaN,NaN,NaN,NaN,NaN
218,219,115,20,20,1,NaN,NaN,NaN,NaN,NaN


**Answers**: Uncomment and run the following to see an answer.

In [11]:
# %load ../answers/Combining_Datasets/ex-concat-1.py
# option 1: reset_index
pd.concat([diet_1, diet_2, diet_3, diet_4]).reset_index(drop=True)

# option 2: parameter
pd.concat([diet_1, diet_2, diet_3, diet_4], ignore_index=True)

In [15]:
# %load ../answers/Combining_Datasets/ex-concat-2.py
pd.concat([diet_1, diet_2], axis=1)

print(f"There are {diet_1.shape[0]} rows in diet_1 and {diet_2.shape[0]} in diet_2")

# Due to the mis-match of the number of rows, there are no values past row 220 
# for diet_2

There are 220 rows in diet_1 and 120 in diet_2


### Adding extra information 

Concatenation is the act of putting two datasets together, either on top of one another or side-by-side. 

Now you will see how to merge data in, for example if you have some new data about the names of the diets:

In [16]:
diet_names = pd.read_csv('../data/diet_names.csv')
diet_names

,diet_id,diet_name
0,1,Prairie’s Choice Backyard Chicken Feed
1,2,Scratch and Peck Chicken Feed
2,3,Manna Pro Layer Feed
3,4,Hiland Naturals Chicken Feed


You may want to add this information to the original data:

In [17]:
chickweight = (
    pd.read_csv('../data/chickweight.csv') 
      .rename(str.lower, axis='columns')
)

If you do this with the `pd.concat()` function, you notice that the output is not what you might expect.

In [18]:
pd.concat([chickweight, diet_names], axis=1)

,rownum,weight,time,chick,diet,diet_id,diet_name
0,1,42,0,1,1,1.0,Prairie’s Choice Backyard Chicken Feed
1,2,51,2,1,1,2.0,Scratch and Peck Chicken Feed
2,3,59,4,1,1,3.0,Manna Pro Layer Feed
3,4,64,6,1,1,4.0,Hiland Naturals Chicken Feed
4,5,76,8,1,1,NaN,NaN
...,...,...,...,...,...,...,...
573,574,175,14,50,4,NaN,NaN
574,575,205,16,50,4,NaN,NaN
575,576,234,18,50,4,NaN,NaN
576,577,264,20,50,4,NaN,NaN


<mark>**Question:** Why do you get the above dataframe as a result?</mark>

<details>
    <summary><font color=blue>Show answer</font></summary>
  
The concatenation occurs on the indexes of the two input dataframes. The index on the `diet_names` only goes up to `3`, where in `chickweight` goes up to `578`.
    
When concatenating it matches the same index values to one another. 
    
</details>

Instead for each value of diet you want to merge on the corresponding diet name.

This is where the **merge** function comes in.

----
<img src="../images/06_Combining_Datasets/join.png" width="200" height="240" align="right"/>

<a id='mj'></a>
## Merging DataFrames

Merging DataFrames is the process of combining two or more similar DataFrames into a single one. 

Merging can be used to add or append variables to a dataset to add information to a DataFrame that exists in another DataFrame.

For example, to **merge** on the diet names, you can use the `pd.merge()` function specifying:
- The names of the DataFrames: `chickweight`, `diet_names`
- On which column to perform the merge from the **left** and **right** dataframes*

*Note: The left DataFrame is the first argument.

In [19]:
pd.merge(
    chickweight,
    diet_names, 
    left_on="diet",
    right_on='diet_id'
)

,rownum,weight,time,chick,diet,diet_id,diet_name
0,1,42,0,1,1,1,Prairie’s Choice Backyard Chicken Feed
1,2,51,2,1,1,1,Prairie’s Choice Backyard Chicken Feed
2,3,59,4,1,1,1,Prairie’s Choice Backyard Chicken Feed
3,4,64,6,1,1,1,Prairie’s Choice Backyard Chicken Feed
4,5,76,8,1,1,1,Prairie’s Choice Backyard Chicken Feed
...,...,...,...,...,...,...,...
573,574,175,14,50,4,4,Hiland Naturals Chicken Feed
574,575,205,16,50,4,4,Hiland Naturals Chicken Feed
575,576,234,18,50,4,4,Hiland Naturals Chicken Feed
576,577,264,20,50,4,4,Hiland Naturals Chicken Feed


Note how there are two diet columns that specify the diet ID numbers. That is because the names of the columns are different. Look what happens if you rename the column before the merge:

In [20]:
pd.merge(
    chickweight.rename(columns={'diet': 'diet_id'}),
    diet_names, 
    left_on='diet_id',
    right_on='diet_id'
)

,rownum,weight,time,chick,diet_id,diet_name
0,1,42,0,1,1,Prairie’s Choice Backyard Chicken Feed
1,2,51,2,1,1,Prairie’s Choice Backyard Chicken Feed
2,3,59,4,1,1,Prairie’s Choice Backyard Chicken Feed
3,4,64,6,1,1,Prairie’s Choice Backyard Chicken Feed
4,5,76,8,1,1,Prairie’s Choice Backyard Chicken Feed
...,...,...,...,...,...,...
573,574,175,14,50,4,Hiland Naturals Chicken Feed
574,575,205,16,50,4,Hiland Naturals Chicken Feed
575,576,234,18,50,4,Hiland Naturals Chicken Feed
576,577,264,20,50,4,Hiland Naturals Chicken Feed


Note that it is not necessary to specify which column you merge on from the left and right since the column names are now same.

Instead you can use the keyword `on=` and use the column named `diet_id`. 

In [23]:
pd.merge(
    chickweight.rename(columns={'diet': 'diet_id'}),
    diet_names, 
    on='diet_id'
)

,rownum,weight,time,chick,diet_id,diet_name
0,1,42,0,1,1,Prairie’s Choice Backyard Chicken Feed
1,2,51,2,1,1,Prairie’s Choice Backyard Chicken Feed
2,3,59,4,1,1,Prairie’s Choice Backyard Chicken Feed
3,4,64,6,1,1,Prairie’s Choice Backyard Chicken Feed
4,5,76,8,1,1,Prairie’s Choice Backyard Chicken Feed
...,...,...,...,...,...,...
573,574,175,14,50,4,Hiland Naturals Chicken Feed
574,575,205,16,50,4,Hiland Naturals Chicken Feed
575,576,234,18,50,4,Hiland Naturals Chicken Feed
576,577,264,20,50,4,Hiland Naturals Chicken Feed


A good thing to consider is what happens when you have two columns that have the same name, which **are not** the column on which to perform the merge. 

To demonstrate, imagine if there was a column in the diet_names DataFrame that was also called `weight`, which specified the amount of feed you receive in a bag of the chicken feed brand.

In [28]:
diet_names = (
    diet_names
    .assign(weight = ['1400g', '1500g', '1350g', '1450g'])
)

diet_names

,diet_id,diet_name,weight
0,1,Prairie’s Choice Backyard Chicken Feed,1400g
1,2,Scratch and Peck Chicken Feed,1500g
2,3,Manna Pro Layer Feed,1350g
3,4,Hiland Naturals Chicken Feed,1450g


What would happen now there are two columns with the same name?

In [25]:
pd.merge(
    chickweight.rename(columns={'diet': 'diet_id'}),
    diet_names, 
    on='diet_id'
).head()

,rownum,weight_x,time,chick,diet_id,diet_name,weight_y
0,1,42,0,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
1,2,51,2,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
2,3,59,4,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
3,4,64,6,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
4,5,76,8,1,1,Prairie’s Choice Backyard Chicken Feed,1400g


Notice that the columns have a suffix. You can control this with the keyword `suffixes=` where the argument is a list of the suffixes you want to use in order of the DataFrames you have passed in previously.

In [ ]:
pd.merge(
    chickweight.rename(columns={'diet': 'diet_id'}),
    diet_names,
    on='diet_id',
    suffixes=['_chick', '_feed']
).head()

,rownum,weight_chick,time,chick,diet_id,diet_name,weight_feed
0,1,42,0,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
1,2,51,2,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
2,3,59,4,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
3,4,64,6,1,1,Prairie’s Choice Backyard Chicken Feed,1400g
4,5,76,8,1,1,Prairie’s Choice Backyard Chicken Feed,1400g


<a id='e1'></a>
## <mark> Assignment : Find the fattest chicken per diet</mark>

1. Use the `.groupby()` method to find the max chick per diet.
2. merge this information to the original chickweight dataset.
3. Find the fattest chickwen per diet by identifying when the value in the weight column is equal to the value in your new column


In [40]:
groupchick = chickweight.groupby(['diet']).agg(max_weight = ('weight','max'))

pd.merge(chickweight, groupchick, on='diet').loc[lambda df: df['weight'] == df['max_weight']]





,rownum,weight,time,chick,diet,max_weight
83,84,305,21,7,1,305
231,232,331,21,21,2,331
399,400,373,21,35,3,373
553,554,322,21,48,4,322


In [42]:
# %load ../answers/Combining_Datasets/ex-fattest-chick-merge.py
chick_by_diet = (
    chickweight
    .groupby('diet')
    .agg(max_weight = ('weight', 'max')
        )
)

(
    pd.merge(
        chickweight,
        chick_by_diet,
        how='left',
        on='diet'
    )
    .loc[lambda df: df['weight']==df['max_weight']]
)